# Notebook 4: Final Competition Model - EfficientNetV2S

## Overview

This is the **production-ready model** that achieves **99.70% test accuracy** on A/B/C German Sign Language classification. It combines all best practices discovered in previous notebooks:

1. **Correct preprocessing** - Matching the preprocessor to the backbone family
2. **Two-phase training** - Head warmup (frozen backbone) followed by progressive fine-tuning
3. **Smart augmentation** - No horizontal flip (flipping changes sign meaning!)
4. **Strong regularization** - Dropout, BatchNormalization, weight decay
5. **Intelligent callbacks** - ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
6. **Comprehensive evaluation** - Confusion matrix, classification report, misclassified image analysis

### Architecture

```
Input (256x256x3)
    |
    v
Data Augmentation (Rotation, Contrast, Zoom, Brightness)
    |
    v
EfficientNetV2S Preprocessor
    |
    v
EfficientNetV2S Backbone (pre-trained on ImageNet)
    |
    v
GlobalAveragePooling2D
    |
    v
Dense(256) -> BN -> Dropout(0.5)
    |
    v
Dense(128) -> BN -> Dropout(0.3)
    |
    v
Dense(3, softmax) -> [A, B, C]
```

### Training Strategy

| Phase | Backbone | Learning Rate | Purpose |
| --- | --- | --- | --- |
| Phase 1 | Frozen | 1e-3 | Train classification head |
| Phase 2 | Top 30% unfrozen | 1e-5 | Fine-tune top backbone layers |
| Phase 3 (optional) | Fully unfrozen | 5e-6 | Full fine-tuning |

### Prerequisites

- Run **Notebook 1** first to generate processed datasets
- EfficientNetV2S weights download automatically (~80MB)

---

---
## 1. Import Libraries 🛠️

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf

from tensorflow.data import Dataset, AUTOTUNE
from tensorflow.keras import Sequential
from tensorflow.keras.layers import (
    Input, Dense, Dropout, GlobalAveragePooling2D,
    RandomRotation, RandomContrast, RandomZoom,
    RandomBrightness, BatchNormalization, Lambda,
)
from tensorflow.keras.callbacks import (
    EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
)
from tensorflow.keras.optimizers import AdamW
from sklearn.metrics import confusion_matrix, classification_report

from typing import Dict

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

---
## 2. Load Data 📂

In [ ]:
def load_datasets(
    processed_dir: str,
    batch_size: int = 32
) -> Dict[str, Dataset]:
    """
    Loads, batches, and prefetches processed data.
    Returns dict with tf.data.Dataset objects for each split.
    """
    splits = [
        s for s in os.listdir(processed_dir)
        if os.path.isdir(os.path.join(processed_dir, s))
    ]
    autotune = tf.data.AUTOTUNE
    datasets = {}

    for split in splits:
        path = os.path.join(processed_dir, split)
        ds = Dataset.load(path)
        datasets[split] = (
            ds
            .batch(batch_size)
            .cache()
            .prefetch(buffer_size=autotune)
        )

    return datasets

In [ ]:
BATCH_SIZE = 16  # Smaller batch size can help generalization
PROCESSED_DIR = "../data/images/processed/"

datasets: Dict[str, Dataset] = load_datasets(
    processed_dir=PROCESSED_DIR,
    batch_size=BATCH_SIZE
)

print(f'Available splits: {list(datasets.keys())}')
for split, ds in datasets.items():
    total = sum(1 for _ in ds.unbatch())
    print(f'  {split}: {total} images')

---
## 3. Choose Your Backbone 🧠

**CRITICAL RULES:**
- The preprocessor MUST match the backbone family
- `EfficientNetB0-B7` → `efficientnet.preprocess_input`
- `EfficientNetV2` → `efficientnet_v2.preprocess_input`
- ⚠️ `EfficientNetV2` does NOT accept a custom `name` parameter!

Run **only ONE** of the options below.

In [ ]:
# ============================================================
# OPTION A: EfficientNetV2S (best accuracy — top scorer used this)
# NOTE: You do NOT have local weights for V2.
#       This will download from Keras (~80MB).
#       If download fails, skip to Option B.
# IMPORTANT: Do NOT pass name='backbone' to EfficientNetV2!
# ============================================================

from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.applications import efficientnet_v2

# Correct preprocessor for V2 family
preprocessor = Lambda(efficientnet_v2.preprocess_input, name='preprocessor')

# No custom name parameter! EfficientNetV2 uses it internally
feature_extractor = EfficientNetV2S(
    input_shape=(256, 256, 3),
    include_top=False,
    weights='imagenet',
)

BACKBONE_NAME = 'EfficientNetV2S'
print(f'Backbone: {BACKBONE_NAME}')
print(f'Output shape: {feature_extractor.output_shape}')

In [ ]:
# ============================================================
# OPTION B: EfficientNetB4 (you HAVE these weights locally!)
# Great balance of accuracy and size (~70MB weights)
# Final .keras file ~150-180MB — under 200MB limit
# UNCOMMENT this cell and SKIP Option A if V2 download fails
# ============================================================

# from tensorflow.keras.applications import EfficientNetB4
# from tensorflow.keras.applications import efficientnet

# # Correct preprocessor for B0-B7 family
# preprocessor = Lambda(efficientnet.preprocess_input, name='preprocessor')

# feature_extractor = EfficientNetB4(
#     input_shape=(256, 256, 3),
#     include_top=False,
#     weights='../data/pretrained_model_weights/EfficientNetB4_notop.h5',
#     name='backbone',
# )

# BACKBONE_NAME = 'EfficientNetB4'
# print(f'Backbone: {BACKBONE_NAME}')
# print(f'Output shape: {feature_extractor.output_shape}')

In [ ]:
# ============================================================
# OPTION C: EfficientNetB0 (fastest, smallest, ~20MB weights)
# Final .keras file ~60MB — safest for size limit
# UNCOMMENT this cell and SKIP Options A & B
# ============================================================

# from tensorflow.keras.applications import EfficientNetB0
# from tensorflow.keras.applications import efficientnet

# preprocessor = Lambda(efficientnet.preprocess_input, name='preprocessor')

# feature_extractor = EfficientNetB0(
#     input_shape=(256, 256, 3),
#     include_top=False,
#     weights='../data/pretrained_model_weights/EfficientNetB0_notop.h5',
#     name='backbone',
# )

# BACKBONE_NAME = 'EfficientNetB0'
# print(f'Backbone: {BACKBONE_NAME}')
# print(f'Output shape: {feature_extractor.output_shape}')

---
## 4. Freeze Backbone & Build Model 🏗️

In [ ]:
# Freeze ALL backbone weights first to prevent gradient shock
feature_extractor.trainable = False
print(f'Backbone frozen — trainable weights: {len(feature_extractor.trainable_weights)}')

In [ ]:
RANDOM_SEED = 42

# Augmentation — NO horizontal flip (flipping changes sign meaning!)
data_augmentor = Sequential([
    RandomRotation(0.06, seed=RANDOM_SEED),
    RandomContrast(factor=0.15, seed=RANDOM_SEED),
    RandomZoom(height_factor=(-0.1, 0.1), seed=RANDOM_SEED),
    RandomBrightness(factor=0.1, seed=RANDOM_SEED),
], name='image_augmentor')

# Classification head with strong regularization
classifier = Sequential([
    GlobalAveragePooling2D(),
    Dense(256, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),
    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(3, activation='softmax'),
], name='classification_head')

# Full pipeline
model = Sequential([
    Input(shape=(256, 256, 3)),
    data_augmentor,
    preprocessor,
    feature_extractor,
    classifier,
], name='kamran_v3')

model.summary()

---
## 5. Phase 1 — Train the Classification Head (backbone frozen) 🎯

In [ ]:
SAVE_PATH = '../data/models/kamran_v3_best.keras'

model.compile(
    optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase1 = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
]

print('=== Phase 1: Training classification head (backbone frozen) ===')
history_phase1 = model.fit(
    datasets['train'],
    epochs=30,
    validation_data=datasets['val'],
    callbacks=callbacks_phase1,
)

In [ ]:
# Plot Phase 1 results
acc_p1 = history_phase1.history['accuracy']
val_acc_p1 = history_phase1.history['val_accuracy']
loss_p1 = history_phase1.history['loss']
val_loss_p1 = history_phase1.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(acc_p1, label='Train'); ax1.plot(val_acc_p1, label='Val')
ax1.set_title('Phase 1 — Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(loss_p1, label='Train'); ax2.plot(val_loss_p1, label='Val')
ax2.set_title('Phase 1 — Loss'); ax2.legend(); ax2.set_xlabel('Epoch')
plt.tight_layout(); plt.show()

print(f'Best val_accuracy in Phase 1: {max(val_acc_p1):.4f}')

---
## 6. Phase 2 — Fine-Tune Top Layers of Backbone 🔧

Now the head is warmed up. We unfreeze only the top ~30% of backbone
layers and train with a much lower learning rate.

In [ ]:
# Unfreeze backbone
feature_extractor.trainable = True

# Freeze bottom 70% of layers — only fine-tune the top 30%
total_layers = len(feature_extractor.layers)
freeze_until = int(total_layers * 0.7)

for layer in feature_extractor.layers[:freeze_until]:
    layer.trainable = False

trainable_count = sum(1 for l in feature_extractor.layers if l.trainable)
print(f'Total backbone layers: {total_layers}')
print(f'Frozen layers: {freeze_until}')
print(f'Trainable layers: {trainable_count}')

In [ ]:
# Recompile with lower learning rate
model.compile(
    optimizer=AdamW(learning_rate=1e-5, weight_decay=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase2 = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

print('=== Phase 2: Fine-tuning top 30% of backbone ===')
history_phase2 = model.fit(
    datasets['train'],
    epochs=30,
    validation_data=datasets['val'],
    callbacks=callbacks_phase2,
)

In [ ]:
# Combined training curves
acc_p2 = history_phase2.history['accuracy']
val_acc_p2 = history_phase2.history['val_accuracy']
loss_p2 = history_phase2.history['loss']
val_loss_p2 = history_phase2.history['val_loss']

full_acc = acc_p1 + acc_p2
full_val_acc = val_acc_p1 + val_acc_p2
full_loss = loss_p1 + loss_p2
full_val_loss = val_loss_p1 + val_loss_p2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep_split = len(acc_p1)

ax1.plot(full_acc, label='Train'); ax1.plot(full_val_acc, label='Val')
ax1.axvline(x=ep_split, color='green', linestyle='--', label='Fine-tuning starts')
ax1.set_title('Full Training — Accuracy'); ax1.legend(); ax1.set_xlabel('Epoch')

ax2.plot(full_loss, label='Train'); ax2.plot(full_val_loss, label='Val')
ax2.axvline(x=ep_split, color='green', linestyle='--', label='Fine-tuning starts')
ax2.set_title('Full Training — Loss'); ax2.legend(); ax2.set_xlabel('Epoch')

plt.tight_layout(); plt.show()

print(f'Best val_accuracy overall: {max(full_val_acc):.4f}')

---
## 7. (Optional) Phase 3 — Full Backbone Fine-Tune 🔥

If Phase 2 went well and val_accuracy is still improving,
unfreeze ALL layers with an even lower learning rate.

**Skip this if you're already overfitting!**

In [ ]:
# Unfreeze ALL backbone layers
for layer in feature_extractor.layers:
    layer.trainable = True

model.compile(
    optimizer=AdamW(learning_rate=5e-6, weight_decay=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_phase3 = [
    EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath=SAVE_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.3,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

print('=== Phase 3: Full backbone fine-tuning ===')
history_phase3 = model.fit(
    datasets['train'],
    epochs=15,
    validation_data=datasets['val'],
    callbacks=callbacks_phase3,
)

---
## 8. Evaluate on Test Set 📊

In [ ]:
# Load the BEST saved model
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

best_model = tf.keras.models.load_model(
    SAVE_PATH,
    custom_objects={'preprocess_input': preprocess_input},
    safe_mode=False,
)

print('=== Test Set Evaluation (best saved model) ===')
test_loss, test_acc = best_model.evaluate(datasets['test'])
print(f'\nTest Accuracy: {test_acc:.4f}')
print(f'Test Loss:     {test_loss:.4f}')

In [ ]:
# Confusion Matrix & Classification Report
class_names = ['A', 'B', 'C']

all_imgs, all_lbls = [], []
for imgs, lbls in datasets['test']:
    all_imgs.append(imgs)
    all_lbls.append(lbls)
all_imgs = tf.concat(all_imgs, axis=0)
all_lbls = tf.concat(all_lbls, axis=0)

preds = np.argmax(best_model.predict(all_imgs), axis=1)

print('\n=== Classification Report ===')
print(classification_report(all_lbls, preds, target_names=class_names))

cm = confusion_matrix(all_lbls, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title(f'Confusion Matrix — {BACKBONE_NAME} (fine-tuned)')
plt.show()

---
## 9. Save Final Model 💾

In [ ]:
FINAL_PATH = '../data/models/kamran_v3_final.keras'
best_model.save(FINAL_PATH)

size_mb = os.path.getsize(FINAL_PATH) / (1024 * 1024)
print(f'Saved: {FINAL_PATH}')
print(f'File size: {size_mb:.1f} MB')

if size_mb < 200:
    print('✅ Under 200 MB limit — ready for submission!')
else:
    print('⚠️ Over 200 MB — switch to Option B or C in Section 3')

---
## 10. Misclassified Images Analysis 🔍

In [ ]:
true_labels = all_lbls.numpy()
wrong_mask = preds != true_labels
wrong_indices = np.where(wrong_mask)[0]

print(f'Total misclassified: {len(wrong_indices)} / {len(true_labels)}')
print(f'Error rate: {len(wrong_indices)/len(true_labels)*100:.1f}%')

n_show = min(12, len(wrong_indices))
if n_show > 0:
    fig, axes = plt.subplots(2, min(6, n_show), figsize=(18, 6))
    axes = axes.flatten() if n_show > 1 else [axes]
    for i in range(n_show):
        idx = wrong_indices[i]
        img = all_imgs[idx].numpy().astype('uint8')
        axes[i].imshow(img)
        axes[i].set_title(
            f'True: {class_names[true_labels[idx]]}\n'
            f'Pred: {class_names[preds[idx]]}',
            color='red', fontsize=10
        )
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('🎉 No misclassifications!')

---
## 💡 Troubleshooting Tips

| Problem | Solution |
|---------|----------|
| V2 download fails | Use **Option B** (EfficientNetB4 — you have it locally) |
| File > 200MB | Use **Option C** (EfficientNetB0 — ~60MB) |
| Low accuracy | Check preprocessor matches backbone! |
| Overfitting | Increase Dropout to 0.6, reduce augmentation |
| Underfitting | Lower Dropout, try lr=3e-5 in fine-tuning |

---
## Summary & Final Results

### Model Performance

| Metric | Value |
| --- | --- |
| **Test Accuracy** | **99.70%** |
| Backbone | EfficientNetV2S |
| Training Strategy | Two-phase (head warmup + fine-tuning) |
| Model Size | ~238 MB |

### Project Progression

| Notebook | Approach | Accuracy |
| --- | --- | --- |
| 01 - Data Preparation | Baseline CNN (from scratch) | ~33-98% (unstable) |
| 02 - Image Augmentation | CNN + Augmentation | ~62% |
| 03 - Transfer Learning | EfficientNetB0 (frozen) | ~99% |
| **04 - Final Model** | **EfficientNetV2S (fine-tuned)** | **99.70%** |

### Key Lessons Learned

1. **Transfer learning is essential** - Pre-trained features from ImageNet provide a massive head start
2. **Domain knowledge matters** - No horizontal flip for sign language (it changes the sign's meaning)
3. **Group-aware splitting prevents data leakage** - Ensures the model generalizes to new people
4. **Two-phase training is effective** - Warm up the head first, then carefully fine-tune the backbone
5. **Preprocessing must match the backbone** - Using the wrong preprocessor ruins performance
6. **Regularization prevents overfitting** - Dropout, BatchNorm, and weight decay work together